# Final 03: Mixed Emotion End-to-End Orchestration and Paper-Ready Outputs

This notebook does not train models. It merges Phase 1 DistilBERT outputs with Phase 2 Llama reasoning outputs, constructs final two-phase predictions, and exports paper-ready metrics, tables, and figures.

In [ ]:
%pip install -q -U pandas numpy scikit-learn matplotlib seaborn openpyxl
print("SETUP COMPLETE. Restart is usually not required for this notebook.")


## Imports and Persistent Output

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support

LABELS = ["Depression", "Neutral", "Happy"]


In [ ]:
# Persistent output configuration.
# The notebook stops if Google Drive is unavailable. This prevents long runs from saving only to /content.
USE_GOOGLE_DRIVE_OUTPUT = True
REQUIRE_PERSISTENT_OUTPUT = True
LOCAL_OUTPUT_ROOT = Path("outputs_final")
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/confidence_guided_llm_reasoning/outputs_final")

OUTPUT_ROOT = LOCAL_OUTPUT_ROOT
DRIVE_OUTPUT_AVAILABLE = False

if USE_GOOGLE_DRIVE_OUTPUT:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
        probe_path = DRIVE_OUTPUT_ROOT / "_drive_write_test.txt"
        probe_path.write_text("ok", encoding="utf-8")
        probe_path.unlink(missing_ok=True)
        OUTPUT_ROOT = DRIVE_OUTPUT_ROOT
        DRIVE_OUTPUT_AVAILABLE = True
        print(f"Google Drive output enabled: {OUTPUT_ROOT}")
    except Exception as exc:
        if REQUIRE_PERSISTENT_OUTPUT:
            raise RuntimeError(
                "Google Drive output is unavailable, so the notebook stopped before running expensive work. "
                "Fix Drive authorization/mount first, or set REQUIRE_PERSISTENT_OUTPUT = False only for a temporary smoke test."
            ) from exc
        print(f"Google Drive output is unavailable ({exc}); falling back to local runtime output.")
        OUTPUT_ROOT = LOCAL_OUTPUT_ROOT
else:
    if REQUIRE_PERSISTENT_OUTPUT:
        raise RuntimeError(
            "USE_GOOGLE_DRIVE_OUTPUT is False while REQUIRE_PERSISTENT_OUTPUT is True. "
            "Turn on Drive output or set REQUIRE_PERSISTENT_OUTPUT = False for a temporary run."
        )

END_TO_END_OUTPUT_DIR = OUTPUT_ROOT / "end_to_end_orchestration"
END_TO_END_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("End-to-end output directory:", END_TO_END_OUTPUT_DIR)


## Input Paths

In [ ]:
PHASE1_PREDICTIONS_PATH = OUTPUT_ROOT / "phase1_distilbert" / "phase1_mixed_emotion_predictions.csv"
LLAMA2_RESULTS_PATH = OUTPUT_ROOT / "phase2_llm_reasoning" / "llama2_cot_routed_mixed_emotion_results.csv"
LLAMA3_RESULTS_PATH = OUTPUT_ROOT / "phase2_llm_reasoning" / "llama3_self_discover_routed_mixed_emotion_results.csv"

print("Phase 1:", PHASE1_PREDICTIONS_PATH)
print("Llama 2:", LLAMA2_RESULTS_PATH)
print("Llama 3:", LLAMA3_RESULTS_PATH)


## Load Outputs

In [ ]:
if not PHASE1_PREDICTIONS_PATH.exists():
    raise FileNotFoundError(f"Missing Phase 1 predictions: {PHASE1_PREDICTIONS_PATH}")
phase1 = pd.read_csv(PHASE1_PREDICTIONS_PATH)
print("phase1", phase1.shape)
display(phase1.head())

llama2 = pd.read_csv(LLAMA2_RESULTS_PATH) if LLAMA2_RESULTS_PATH.exists() else pd.DataFrame()
llama3 = pd.read_csv(LLAMA3_RESULTS_PATH) if LLAMA3_RESULTS_PATH.exists() else pd.DataFrame()
print("llama2", llama2.shape)
print("llama3", llama3.shape)

required_phase1 = {"example_id", "text", "target_label", "phase1_label", "phase1_confidence", "phase1_accepted", "phase1_routed"}
missing = required_phase1 - set(phase1.columns)
if missing:
    raise ValueError(f"Phase 1 file missing columns: {missing}")


## Merge and Build End-to-End Predictions

In [ ]:
def normalize_label(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    for label in LABELS:
        if text.lower() == label.lower():
            return label
    for label in LABELS:
        if label.lower() in text.lower():
            return label
    return np.nan

base = phase1.copy()
base["phase1_label"] = base["phase1_label"].map(normalize_label)
base["target_label"] = base["target_label"].map(normalize_label)
base["phase1_routed"] = base["phase1_routed"].astype(str).str.lower().isin(["true", "1", "yes"])
base["phase1_accepted"] = ~base["phase1_routed"]

if not llama2.empty:
    keep = ["example_id", "LLaMA2_final_label"]
    base = base.merge(llama2[keep].drop_duplicates("example_id", keep="last"), on="example_id", how="left")
    base["LLaMA2_final_label"] = base["LLaMA2_final_label"].map(normalize_label)
else:
    base["LLaMA2_final_label"] = np.nan

if not llama3.empty:
    keep = ["example_id", "LLaMA3_final_label"]
    base = base.merge(llama3[keep].drop_duplicates("example_id", keep="last"), on="example_id", how="left")
    base["LLaMA3_final_label"] = base["LLaMA3_final_label"].map(normalize_label)
else:
    base["LLaMA3_final_label"] = np.nan

base["final_label_llama2"] = np.where(base["phase1_routed"] & base["LLaMA2_final_label"].notna(), base["LLaMA2_final_label"], base["phase1_label"])
base["final_source_llama2"] = np.where(base["phase1_routed"] & base["LLaMA2_final_label"].notna(), "llama2_cot", "phase1")
base["final_label_llama3"] = np.where(base["phase1_routed"] & base["LLaMA3_final_label"].notna(), base["LLaMA3_final_label"], base["phase1_label"])
base["final_source_llama3"] = np.where(base["phase1_routed"] & base["LLaMA3_final_label"].notna(), "llama3_self_discover", "phase1")

base["is_correct_phase1"] = base["phase1_label"] == base["target_label"]
base["is_correct_llama2_e2e"] = base["final_label_llama2"] == base["target_label"]
base["is_correct_llama3_e2e"] = base["final_label_llama3"] == base["target_label"]

out_path = END_TO_END_OUTPUT_DIR / "mixed_emotion_end_to_end_results.csv"
base.to_csv(out_path, index=False)
print("Saved:", out_path)
display(base.head())


## Metrics and Correction Analysis

In [ ]:
def metrics_for(df, pred_col, name):
    valid = df.dropna(subset=["target_label", pred_col]).copy()
    if valid.empty:
        return {"model": name, "rows": 0, "accuracy": np.nan, "macro_precision": np.nan, "macro_recall": np.nan, "macro_f1": np.nan}
    p, r, f1, _ = precision_recall_fscore_support(valid["target_label"], valid[pred_col], labels=LABELS, average="macro", zero_division=0)
    return {"model": name, "rows": len(valid), "accuracy": accuracy_score(valid["target_label"], valid[pred_col]), "macro_precision": p, "macro_recall": r, "macro_f1": f1}

metrics = [metrics_for(base, "phase1_label", "Phase 1 DistilBERT only")]
if base["LLaMA2_final_label"].notna().any():
    metrics.append(metrics_for(base[base["phase1_routed"]], "LLaMA2_final_label", "Llama 2 routed only"))
    metrics.append(metrics_for(base, "final_label_llama2", "End-to-end with Llama 2"))
if base["LLaMA3_final_label"].notna().any():
    metrics.append(metrics_for(base[base["phase1_routed"]], "LLaMA3_final_label", "Llama 3 routed only"))
    metrics.append(metrics_for(base, "final_label_llama3", "End-to-end with Llama 3"))
metrics_df = pd.DataFrame(metrics)
metrics_path = END_TO_END_OUTPUT_DIR / "end_to_end_metrics_summary.csv"
metrics_df.to_csv(metrics_path, index=False)
display(metrics_df)

correction_rows = []
for model_name, final_col, phase2_col in [("llama2", "final_label_llama2", "LLaMA2_final_label"), ("llama3", "final_label_llama3", "LLaMA3_final_label")]:
    if phase2_col not in base or not base[phase2_col].notna().any():
        continue
    routed = base[base["phase1_routed"]].copy()
    corrected = (~routed["is_correct_phase1"]) & (routed[phase2_col] == routed["target_label"])
    introduced = (routed["is_correct_phase1"]) & (routed[phase2_col].notna()) & (routed[phase2_col] != routed["target_label"])
    remaining = (~routed["is_correct_phase1"]) & (routed[phase2_col] != routed["target_label"])
    correction_rows.append({
        "phase2_model": model_name,
        "routed_rows": len(routed),
        "phase1_errors_in_routed": int((~routed["is_correct_phase1"]).sum()),
        "corrected_errors": int(corrected.sum()),
        "introduced_errors": int(introduced.sum()),
        "remaining_routed_errors": int(remaining.sum()),
    })
correction_df = pd.DataFrame(correction_rows)
correction_path = END_TO_END_OUTPUT_DIR / "phase2_correction_analysis.csv"
correction_df.to_csv(correction_path, index=False)
display(correction_df)

routing_summary = pd.DataFrame([{
    "total_rows": len(base),
    "accepted_by_phase1": int(base["phase1_accepted"].sum()),
    "routed_to_phase2": int(base["phase1_routed"].sum()),
    "coverage": float(base["phase1_accepted"].mean()),
    "routing_rate": float(base["phase1_routed"].mean()),
    "routing_threshold": float(base["routing_threshold"].dropna().iloc[0]) if "routing_threshold" in base and base["routing_threshold"].notna().any() else np.nan,
    "temperature": float(base["temperature"].dropna().iloc[0]) if "temperature" in base and base["temperature"].notna().any() else np.nan,
}])
routing_summary.to_csv(END_TO_END_OUTPUT_DIR / "routing_coverage_table.csv", index=False)
display(routing_summary)


## Paper-Ready Figures and Tables

In [ ]:
def save_cm(df, pred_col, title, filename):
    valid = df.dropna(subset=["target_label", pred_col])
    cm = confusion_matrix(valid["target_label"], valid[pred_col], labels=LABELS)
    plt.figure(figsize=(5.2, 4.2))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=LABELS, yticklabels=LABELS)
    plt.xlabel("Predicted")
    plt.ylabel("Target")
    plt.title(title)
    plt.tight_layout()
    path = END_TO_END_OUTPUT_DIR / filename
    plt.savefig(path, dpi=220)
    plt.show()
    return path

save_cm(base, "phase1_label", "Phase 1 DistilBERT", "confusion_matrix_phase1.png")
if base["LLaMA2_final_label"].notna().any():
    save_cm(base, "final_label_llama2", "End-to-End with Llama 2", "confusion_matrix_llama2_e2e.png")
if base["LLaMA3_final_label"].notna().any():
    save_cm(base, "final_label_llama3", "End-to-End with Llama 3", "confusion_matrix_llama3_e2e.png")

with pd.ExcelWriter(END_TO_END_OUTPUT_DIR / "paper_ready_tables.xlsx") as writer:
    metrics_df.to_excel(writer, index=False, sheet_name="metrics_summary")
    routing_summary.to_excel(writer, index=False, sheet_name="routing_coverage")
    correction_df.to_excel(writer, index=False, sheet_name="correction_analysis")
    pd.crosstab(base["target_label"], base["phase1_label"]).to_excel(writer, sheet_name="cm_phase1")
    if base["LLaMA2_final_label"].notna().any():
        pd.crosstab(base["target_label"], base["final_label_llama2"]).to_excel(writer, sheet_name="cm_llama2_e2e")
    if base["LLaMA3_final_label"].notna().any():
        pd.crosstab(base["target_label"], base["final_label_llama3"]).to_excel(writer, sheet_name="cm_llama3_e2e")
print("Saved paper-ready workbook:", END_TO_END_OUTPUT_DIR / "paper_ready_tables.xlsx")


## Final Export

In [ ]:
OUTPUT_DIR_FOR_EXPORT = END_TO_END_OUTPUT_DIR
EXPORT_ZIP_NAME = "mixed_emotion_end_to_end_paper_outputs"
FINAL_MODEL_DIR = None
# Final export / download cell.
# This creates one zip file containing all available outputs from this notebook.
from pathlib import Path
import zipfile

files_to_zip = []
for pattern in ["*.csv", "*.json", "*.png", "*.xlsx"]:
    files_to_zip.extend(sorted(OUTPUT_DIR_FOR_EXPORT.glob(pattern)))

# Include saved model files if present, but avoid adding huge checkpoint internals repeatedly.
model_dir = globals().get("FINAL_MODEL_DIR")
if model_dir is not None and Path(model_dir).exists():
    for p in Path(model_dir).glob("*"):
        if p.is_file():
            files_to_zip.append(p)

print("Files found for export:")
for p in files_to_zip:
    print(f"- {p} | {p.stat().st_size:,} bytes")

if not files_to_zip:
    print("No output files found yet.")
else:
    zip_path = OUTPUT_DIR_FOR_EXPORT / f"{EXPORT_ZIP_NAME}.zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for p in files_to_zip:
            zf.write(p, arcname=p.relative_to(OUTPUT_DIR_FOR_EXPORT) if p.is_relative_to(OUTPUT_DIR_FOR_EXPORT) else p.name)
    print(f"Saved zip: {zip_path} | {zip_path.stat().st_size:,} bytes")
    if DRIVE_OUTPUT_AVAILABLE:
        print("Persistent Drive copy is available here:")
        print(zip_path)
    try:
        from google.colab import files
        files.download(str(zip_path))
    except Exception as exc:
        print(f"Automatic browser download was not started: {exc}")
